In [ ]:
from google.adk.agents import Agent # 正确导入
from google.adk.tools.base_tool import BaseTool
from google.adk.tools.tool_context import ToolContext
from typing import Optional, Dict, Any

def validate_tool_params(
    tool: BaseTool,
    args: Dict[str, Any],
    tool_context: ToolContext # 正确的签名，删除CallbackContext
    ) -> Optional[Dict]:
    """
    Validates tool arguments before execution.
    For example, checks if the user ID in the arguments matches the one in the session state.
    """
    print(f"Callback triggered for tool: {tool.name}, args: {args}")

    # 通过tool_context正确访问状态
    expected_user_id = tool_context.state.get("session_user_id")
    actual_user_id_in_args = args.get("user_id_param")

    if actual_user_id_in_args and actual_user_id_in_args != expected_user_id:
        print(f"Validation Failed: User ID mismatch for tool '{tool.name}'.")
        # 通过返回字典来阻止工具执行
        return {
            "status": "error",
            "error_message": f"Tool call blocked: User ID validation failed for security reasons."
        }

    # 允许工具继续执行
    print(f"Callback validation passed for tool '{tool.name}'.")
    return None

# 使用记录的类设置代理
root_agent = Agent( # 使用记录的 Agent 类
    model='gemini-2.0-flash-exp', # 使用指南中的型号名称
    name='root_agent',
    instruction="You are a root agent that validates tool calls.",
    before_tool_callback=validate_tool_params, # 分配更正后的回调
    tools = [
      # ...工具功能或工具实例列表...
    ]
)